# Matrix Decomposition and Visualization of Induction Heads

#### Question - what is the interpretation of each of the following matrices?

#### $W_{OV}^{h}$

<details>
<summary>Answer</summary>

$W_{OV}^{h}$ has size $(d_\text{model}, d_\text{model})$, it is a linear map describing **what information gets moved from source to destination, in the residual stream.**

In other words, if $x$ is a vector in the residual stream, then $x^T W_{OV}^{h}$ is the vector written to the residual stream at the destination position, if the destination token only pays attention to the source token at the position of the vector $x$.
</details>

#### $W_E W_{OV}^h W_U$

<details>
<summary>Hint</summary>

If $A$ is the one-hot encoding for token `A` (i.e. the vector with zeros everywhere except for a one in the position corresponding to token `A`), then think about what $A^T W_E W_{OV}^h W_U$ represents. You can evaluate this expression from left to right (e.g. start with thinking about what $A^T W_E$ represents, then multiply by the other two matrices).
</details>
<details>
<summary>Answer</summary>

$W_E W_{OV}^h W_U$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a linear map describing **what information gets moved from source to destination, in a start-to-end sense.**

If $A$ is the one-hot encoding for token `A`, then:

* $A^T W_E$ is the embedding vector for `A`.
* $A^T W_E W_{OV}^h$ is the vector which would get written to the residual stream at the destination position, if the destination token only pays attention to `A`.
* $A^T W_E W_{OV}^h W_U$ is the unembedding of this vector, i.e. the thing which gets added to the final logits.

</details>

#### $W_{QK}^{h}$

<details>
<summary>Answer</summary>

$W_{QK}^{h}$ has size $(d_\text{model}, d_\text{model})$, it is a bilinear form describing **where information is moved to and from** in the residual stream (i.e. which residual stream vectors attend to which others).

$x_i^T W_{QK}^h x_j = (x_i^T W_Q^h) (x_j^T W_K^h)^T$ is the attention score paid by token $i$ to token $j$.
</details>

#### $W_E W_{QK}^h W_E^T$

<details>
<summary>Answer</summary>

$W_E W_{QK}^h W_E^T$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a bilinear form describing **where information is moved to and from**, among words in our vocabulary (i.e. which tokens pay attention to which others).

If $A$ and $B$ are one-hot encodings for tokens `A` and `B`, then $A^T W_E W_{QK}^h W_E^T B$ is the attention score paid by token `A` to token `B`:

$$
A^T \, W_E\, W_{QK}^{h}\, W_E^T \, B = \underbrace{(A^T W_E W_Q^{h})}_{\text{query for token } A}  \underbrace{(B^T W_E W_K^{h})^T}_{\text{key for token }B}
$$
</details>

#### $W_{pos} W_{QK}^h W_{pos}^T$

<details>
<summary>Answer</summary>

$W_{pos} W_{QK}^h W_{pos}^T$ has size $(n_\text{ctx}, n_\text{ctx})$, it is a bilinear form describing **where information is moved to and from**, among tokens in our context (i.e. which token positions pay attention to other positions).

If $i$ and $j$ are one-hot encodings for positions `i` and `j` (in other words they are just the ith and jth basis vectors), then $i^T W_{pos} W_{QK}^h W_{pos}^T j$ is the attention score paid by the token with position `i` to the token with position `j`:

$$
i^T \, W_{pos}\, W_{QK}^{h}\, W_{pos}^T \, j = \underbrace{(i^T W_{pos} W_Q^{h})}_{\text{query for i-th token}}  \underbrace{(j^T W_{pos} W_K^{h})^T}_{\text{key for j-th token}}
$$

</details>

#### $W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T$

where $h_1$ is in an earlier layer than $h_2$.

<details>
<summary>Hint</summary>

This matrix is best seen as a bilinear form of size $(d_\text{vocab}, d_\text{vocab})$. The $(A, B)$-th element is:

$$
(A^T W_E W_{OV}^{h_1}) W_{QK}^{h_2} (B^T W_E)^T
$$
</details>

<details>
<summary>Answer</summary>

$W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a bilinear form describing where information is moved to and from in head $h_2$, given that the **query-side vector** is formed from the output of head $h_1$. In other words, this is an instance of **Q-composition**.

If $A$ and $B$ are one-hot encodings for tokens `A` and `B`, then $A^T W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T B$ is the attention score paid **to** token `B`, **by** any token which attended strongly to an `A`-token in head $h_1$.

---

To further break this down, if it still seems confusing:

$$
\begin{aligned}
A^T \, W_E\, W_{OV}^{h_1} W_{QK}^{h_2}\, W_E^T \, B &= \underbrace{(A^T W_E W_{OV}^{h_1}W_Q^{h_2})}_{\text{query of token which attended to A}}  \underbrace{(B^T W_E W_K^{h_2})^T}_\text{key of token B} \\
\end{aligned}
$$

---

Note that the actual attention score will be a sum of multiple terms, not just this one (in fact, we'd have a different term for every combination of query and key input). But this term describes the **particular contribution** to the attention score from this combination of query and key input, and it might be the case that this term is the only one that matters (i.e. all other terms don't much affect the final probabilities). We'll see something exactly like this later on.
</details>